In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

In [0]:
BASE_DIR = f"/Volumes/pk_big_data/raw/zenodo_files"

CATALOG = "pk_big_data"
SCHEMA = "curated"

In [0]:
display(dbutils.fs.ls(BASE_DIR))

path,name,size,modificationTime
dbfs:/Volumes/pk_big_data/raw/zenodo_files/airports.parquet,airports.parquet,151553,1795172320000
dbfs:/Volumes/pk_big_data/raw/zenodo_files/flightlist_final.parquet,flightlist_final.parquet,98418,1795172320000
dbfs:/Volumes/pk_big_data/raw/zenodo_files/flightlist_rank.parquet,flightlist_rank.parquet,69002,1795172320000
dbfs:/Volumes/pk_big_data/raw/zenodo_files/flightlist_train.parquet,flightlist_train.parquet,361072,1795172321000
dbfs:/Volumes/pk_big_data/raw/zenodo_files/flights_final/,flights_final/,0,1795172667000
dbfs:/Volumes/pk_big_data/raw/zenodo_files/flights_rank/,flights_rank/,0,1795172676000
dbfs:/Volumes/pk_big_data/raw/zenodo_files/flights_train/,flights_train/,0,1795172608000
dbfs:/Volumes/pk_big_data/raw/zenodo_files/fuel_final.parquet,fuel_final.parquet,1030742,1795172321000
dbfs:/Volumes/pk_big_data/raw/zenodo_files/fuel_final_submit.parquet,fuel_final_submit.parquet,1528389,1795172322000
dbfs:/Volumes/pk_big_data/raw/zenodo_files/fuel_rank.parquet,fuel_rank.parquet,683090,1795172322000


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    LongType, StringType, DoubleType
)

flights_final_schema = StructType([
    StructField("timestamp", LongType(), True),
    StructField("flight_id", StringType(), True),
    StructField("typecode", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("altitude", DoubleType(), True),
    StructField("groundspeed", DoubleType(), True),
    StructField("track", DoubleType(), True),
    StructField("vertical_rate", DoubleType(), True),
    StructField("mach", DoubleType(), True),
    StructField("TAS", DoubleType(), True),
    StructField("CAS", DoubleType(), True),
    StructField("source", StringType(), True)
])


flights_rank_schema = StructType([
    StructField("timestamp", LongType(), True),
    StructField("flight_id", StringType(), True),
    StructField("typecode", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("altitude", DoubleType(), True),
    StructField("groundspeed", DoubleType(), True),
    StructField("track", DoubleType(), True),
    StructField("vertical_rate", DoubleType(), True),
    StructField("mach", DoubleType(), True),
    StructField("TAS", DoubleType(), True),
    StructField("CAS", DoubleType(), True),
    StructField("source", StringType(), True)
])


flights_train_schema = StructType([
    StructField("timestamp", LongType(), True),
    StructField("flight_id", StringType(), True),
    StructField("typecode", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("altitude", DoubleType(), True),
    StructField("groundspeed", DoubleType(), True),
    StructField("track", DoubleType(), True),
    StructField("vertical_rate", DoubleType(), True),
    StructField("mach", DoubleType(), True),
    StructField("TAS", DoubleType(), True),
    StructField("CAS", DoubleType(), True),
    StructField("source", StringType(), True)
])

fuel_final_schema = StructType([
    StructField("idx", LongType(), True),
    StructField("flight_id", StringType(), True),
    StructField("start", LongType(), True),
    StructField("end", LongType(), True),
    StructField("fuel_kg", DoubleType(), True)
])


fuel_final_submit_schema = StructType([
    StructField("idx", LongType(), True),
    StructField("flight_id", StringType(), True),
    StructField("start", LongType(), True),
    StructField("end", LongType(), True),
    StructField("fuel_kg", DoubleType(), True)
])

fuel_rank_schema = StructType([
    StructField("idx", LongType(), True),
    StructField("flight_id", StringType(), True),
    StructField("start", LongType(), True),
    StructField("end", LongType(), True),
    StructField("fuel_kg", DoubleType(), True)
])


fuel_rank_submit_schema = StructType([
    StructField("idx", LongType(), True),
    StructField("flight_id", StringType(), True),
    StructField("start", LongType(), True),
    StructField("end", LongType(), True),
    StructField("fuel_kg", DoubleType(), True)
])


fuel_train_schema = StructType([
    StructField("idx", LongType(), True),
    StructField("flight_id", StringType(), True),
    StructField("start", LongType(), True),
    StructField("end", LongType(), True),
    StructField("fuel_kg", DoubleType(), True)
])

In [0]:
df_airports = spark.read.parquet(f"{BASE_DIR}/airports.parquet")
df_flightlist_final = spark.read.parquet(f"{BASE_DIR}/flightlist_final.parquet")
df_flightlist_rank = spark.read.parquet(f"{BASE_DIR}/flightlist_rank.parquet")
df_flightlist_train = spark.read.parquet(f"{BASE_DIR}/flightlist_train.parquet")

df_flights_final = (
    spark.read.schema(flights_final_schema)
    .parquet(f"{BASE_DIR}/flights_final/")
    .withColumn("timestamp_converted", F.expr("timestamp_micros(timestamp DIV 1000)"))
)

df_flights_rank = (
    spark.read.schema(flights_rank_schema)
    .parquet(f"{BASE_DIR}/flights_rank/")
    .withColumn("timestamp_converted", F.expr("timestamp_micros(timestamp DIV 1000)"))
)


df_flights_train = (
    spark.read.schema(flights_train_schema)
    .parquet(f"{BASE_DIR}/flights_train/")
    .withColumn("timestamp_converted", F.expr("timestamp_micros(timestamp DIV 1000)"))
)


df_fuel_final = (
    spark.read.schema(fuel_final_schema)
    .parquet(f"{BASE_DIR}/fuel_final.parquet")
    .withColumn("start_converted", F.expr("timestamp_micros(start DIV 1000)"))
    .withColumn("end_converted", F.expr("timestamp_micros(end DIV 1000)"))
)

df_fuel_final_submit = (
    spark.read.schema(fuel_final_submit_schema)
    .parquet(f"{BASE_DIR}/fuel_final_submit.parquet")
    .withColumn("start_converted", F.expr("timestamp_micros(start DIV 1000)"))
    .withColumn("end_converted", F.expr("timestamp_micros(end DIV 1000)"))
)

df_fuel_rank = (
    spark.read.schema(fuel_rank_schema)
    .parquet(f"{BASE_DIR}/fuel_rank.parquet")
    .withColumn("start_converted", F.expr("timestamp_micros(start DIV 1000)"))
    .withColumn("end_converted", F.expr("timestamp_micros(end DIV 1000)"))
)

df_fuel_rank_submit = (
    spark.read.schema(fuel_rank_submit_schema)
    .parquet(f"{BASE_DIR}/fuel_rank_submit.parquet")
    .withColumn("start_converted", F.expr("timestamp_micros(start DIV 1000)"))
    .withColumn("end_converted", F.expr("timestamp_micros(end DIV 1000)"))
)

df_fuel_train = (
    spark.read.schema(fuel_train_schema)
    .parquet(f"{BASE_DIR}/fuel_train.parquet")
    .withColumn("start_converted", F.expr("timestamp_micros(start DIV 1000)"))
    .withColumn("end_converted", F.expr("timestamp_micros(end DIV 1000)"))
)

Podczas wczytywania części zbiorów danych w formacie Parquet wystąpił błąd `PARQUET_TYPE_ILLEGAL`, spowodowany brakiem obsługi kolumn typu `TIMESTAMP(NANOS)` o precyzji nanosekundowej. W rezultacie niektórych plików nie można było odczytać standardową metodą **schema-on-read**, nawet po ręcznym zdefiniowaniu schematu za pomocą `StructType`. Próbowano również rozwiązać problem poprzez zmianę konfiguracji Sparka, jednak środowisko dla wersji Sparka 4.2.0 nie udostępniało wymaganych ustawień. Konieczne było zatem wczytanie kolumn jako LongType i konwersja do Timestamp.

In [0]:
df_flights_train.display()

timestamp,flight_id,typecode,latitude,longitude,altitude,groundspeed,track,vertical_rate,mach,TAS,CAS,source,timestamp_converted
1749615988230000019,prc785525970,A359,33.63473665916313,-84.42375183105469,825.0936,145.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:28.230Z
1749615988789999962,prc785525970,A359,33.634735107421875,-84.42421426578443,825.0936,146.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:28.789Z
1749615989187000036,prc785525970,A359,33.634735107421875,-84.42455058195151,825.0936,148.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:29.187Z
1749615989861000061,prc785525970,A359,33.634735107421875,-84.42505505620215,825.0936,152.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:29.861Z
1749615991375000000,prc785525970,A359,33.634735107421875,-84.42640032087053,825.0936,156.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:31.375Z
1749615992947999954,prc785525970,A359,33.63473665916313,-84.42781448364258,825.0936,161.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:32.947Z
1749615994233999968,prc785525970,A359,33.634735107421875,-84.4288666394292,825.0936,166.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:34.233Z
1749615994617000103,prc785525970,A359,33.63473665916313,-84.42930221557617,825.0936,166.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:34.617Z
1749615995242000103,prc785525970,A359,33.63473665916313,-84.429931640625,825.0936,169.0,270.0,192.0,null,null,null,adsb,2025-06-11T04:26:35.242Z
1749615995765000105,prc785525970,A359,33.634735107421875,-84.43043611487565,825.0936,170.0,270.0,192.0,null,null,null,adsb,2025-06-11T04:26:35.765Z


In [0]:
df_airports.display() 

icao,longitude,latitude,elevation
HCAD,46.6375,6.095802,1001
OAAK,65.206746,36.942659,900
OALG,69.023102,33.9865,6333
OAMK,67.848602,32.879601,6600
OAPJ,67.020445,34.391726,8800
OATW,64.42542,33.54309,7370
AGAF,160.948611,-9.191389,23
AGAR,161.979546547,-9.86054358262,40
AGAT,161.011002,-8.87333,null
AGBA,156.70599365234375,-7.912779808044434,null


In [0]:
df_flightlist_final.display()

flight_date,aircraft_type,takeoff,landed,origin_icao,origin_name,destination_icao,destination_name,flight_id
2025-10-01,A20N,2025-10-01T01:12:11.000,2025-10-01T04:51:56.000,MGGT,Guatemala City La Aurora International Airport,KIAD,Washington Dulles International Airport,prc814299936
2025-10-01,A20N,2025-10-01T01:42:49.000,2025-10-01T05:46:35.000,MSLP,San Salvador International Airport,KJFK,New York John F. Kennedy International Airport,prc814304455
2025-10-01,A320,2025-10-01T02:34:16.000,2025-10-01T06:18:14.000,MSLP,San Salvador International Airport,KIAD,Washington Dulles International Airport,prc814312041
2025-10-01,A20N,2025-10-01T02:52:38.000,2025-10-01T07:52:40.000,SEGU,Guayaquil International Airport,SAEZ,Buenos Aires Ministro Pistarini International Airport,prc814314137
2025-10-01,A320,2025-10-01T04:38:48.000,2025-10-01T08:19:22.000,VTBD,Bangkok Don Mueang International Airport,WIII,Jakarta Soekarno Hatta International Airport,prc814321568
2025-10-01,A20N,2025-10-01T04:10:46.000,2025-10-01T09:40:08.000,CYYZ,Toronto Pearson International Airport,SKBO,Bogota El Dorado International Airport,prc814322542
2025-10-01,A320,2025-10-01T04:01:53.000,2025-10-01T08:53:55.000,SKRG,Medellin Jose Maria Cordova International Airport,KJFK,New York John F. Kennedy International Airport,prc814322739
2025-10-01,A20N,2025-10-01T04:04:34.000,2025-10-01T07:50:18.000,MSLP,San Salvador International Airport,KIAD,Washington Dulles International Airport,prc814323273
2025-10-01,A320,2025-10-01T04:15:48.000,2025-10-01T08:56:28.000,MMGL,Guadalajara International Airport,KEWR,New York Newark Liberty International Airport,prc814323506
2025-10-01,A20N,2025-10-01T04:50:20.000,2025-10-01T08:57:03.000,MSLP,San Salvador International Airport,KJFK,New York John F. Kennedy International Airport,prc814329273


In [0]:
df_flightlist_rank.display()

flight_date,aircraft_type,takeoff,landed,origin_icao,origin_name,destination_icao,destination_name,flight_id
2025-09-01,A320,2025-09-01T00:46:27.000,2025-09-01T04:13:27.000,MMGL,Guadalajara International Airport,KOAK,Oakland International Airport,prc806615763
2025-09-01,A20N,2025-09-01T00:40:28.000,2025-09-01T04:36:17.000,MSLP,San Salvador International Airport,KIAD,Washington Dulles International Airport,prc806615831
2025-09-01,A320,2025-09-01T00:58:47.000,2025-09-01T04:44:01.000,MMMM,Morelia International Airport,KOAK,Oakland International Airport,prc806618686
2025-09-01,A20N,2025-09-01T01:11:08.000,2025-09-01T07:19:51.000,SEGU,Guayaquil International Airport,KJFK,New York John F. Kennedy International Airport,prc806619863
2025-09-01,A320,2025-09-01T01:13:38.000,2025-09-01T05:04:01.000,MMMM,Morelia International Airport,KSMF,Sacramento International Airport,prc806620863
2025-09-01,A320,2025-09-01T03:04:18.000,2025-09-01T06:32:42.000,MMGL,Guadalajara International Airport,KSMF,Sacramento International Airport,prc806633982
2025-09-01,A20N,2025-09-01T04:35:06.000,2025-09-01T09:39:55.000,SKBO,Bogota El Dorado International Airport,KJFK,New York John F. Kennedy International Airport,prc806638881
2025-09-01,A20N,2025-09-01T05:06:42.000,2025-09-01T08:40:51.000,MMMM,Morelia International Airport,KORD,Chicago O'Hare International Airport,prc806642601
2025-09-01,A359,2025-09-01T05:59:19.000,2025-09-01T20:15:45.000,KLAX,Los Angeles International Airport,YSSY,Sydney Kingsford Smith Airport,prc806646128
2025-09-01,A21N,2025-09-01T05:03:26.000,2025-09-01T08:02:32.000,ZSPD,Shanghai Pudong International Airport,ZJSY,Sanya Phoenix International Airport,prc806649811


In [0]:
df_flightlist_train.display()

flight_date,aircraft_type,takeoff,landed,origin_icao,origin_name,destination_icao,destination_name,flight_id
2025-04-12,B789,2025-04-12T16:15:35.000,2025-04-13T04:33:52.000,WMKK,Kuala Lumpur International Airport,EHAM,Amsterdam Schiphol Airport,prc770822360
2025-04-12,A359,2025-04-12T16:40:41.000,2025-04-13T03:10:15.000,ZSPD,Shanghai Pudong International Airport,NZAA,Auckland Airport,prc770831136
2025-04-12,A359,2025-04-12T17:01:38.000,2025-04-13T06:10:27.000,ZSPD,Shanghai Pudong International Airport,LEMD,Madrid Barajas Airport,prc770835414
2025-04-12,B789,2025-04-12T17:53:11.000,2025-04-13T05:05:13.000,ZSPD,Shanghai Pudong International Airport,LHBP,Budapest Ferenc Liszt International Airport,prc770844923
2025-04-12,B789,2025-04-12T18:05:27.000,2025-04-13T06:23:52.000,ZSPD,Shanghai Pudong International Airport,LFML,Marseille Provence Airport,prc770847190
2025-04-12,B788,2025-04-12T19:38:42.000,2025-04-13T04:59:13.000,SKBO,Bogota El Dorado International Airport,LFPG,Paris Charles de Gaulle Airport,prc770852254
2025-04-12,A359,2025-04-12T20:06:20.000,2025-04-13T12:08:17.000,FAOR,Johannesburg OR Tambo International Airport,KATL,Atlanta Hartsfield-Jackson International Airport,prc770853645
2025-04-12,B788,2025-04-12T20:36:07.000,2025-04-13T05:19:27.000,SKBO,Bogota El Dorado International Airport,LEMD,Madrid Barajas Airport,prc770859206
2025-04-12,B789,2025-04-12T19:38:11.000,2025-04-12T22:22:46.000,NZAA,Auckland Airport,YSSY,Sydney Kingsford Smith Airport,prc770860078
2025-04-12,B789,2025-04-12T20:04:12.000,2025-04-13T05:38:45.000,VOBL,Bengaluru Kempegowda International Airport,EHAM,Amsterdam Schiphol Airport,prc770864956


In [0]:
df_flights_final.display()

timestamp,flight_id,typecode,latitude,longitude,altitude,groundspeed,track,vertical_rate,mach,TAS,CAS,source,timestamp_converted
1759556048391999960,prc815137578,A359,39.234100341796875,116.58942181131114,34100.0,544.0,20.235271166382038,0.0,null,null,null,adsb,2025-10-04T05:34:08.391Z
1759556048828000069,prc815137578,A359,39.23520362983316,116.5899658203125,34100.0,544.0,20.235271166382038,0.0,null,null,null,adsb,2025-10-04T05:34:08.828Z
1759556049263999939,prc815137578,A359,39.23613467458951,116.59039306640625,34100.0,544.0,20.235271166382038,0.0,null,null,null,adsb,2025-10-04T05:34:09.263Z
1759556052282000065,prc815137578,A359,39.24311751026218,116.59375,34100.0,544.0,20.334113032500525,0.0,null,null,null,adsb,2025-10-04T05:34:12.282Z
1759556055154999971,prc815137578,A359,39.24991413698358,116.59698486328125,34100.0,543.0,20.37078251263994,0.0,null,null,null,adsb,2025-10-04T05:34:15.154Z
1759556056069999933,prc815137578,A359,39.25238140558792,116.59814453125,34100.0,543.0,20.37078251263994,0.0,null,null,null,adsb,2025-10-04T05:34:16.069Z
1759556057158999920,prc815137578,A359,39.25465393066406,116.5992139733356,34100.0,null,null,null,null,null,null,adsb,2025-10-04T05:34:17.158Z
1759556060125999928,prc815137578,A359,39.261784957627114,116.6026611328125,34100.0,null,null,null,null,null,null,adsb,2025-10-04T05:34:20.125Z
1759556061216000080,prc815137578,A359,39.26443843518273,116.6038818359375,34100.0,null,null,null,null,null,null,adsb,2025-10-04T05:34:21.216Z
1759556062180999994,prc815137578,A359,39.266510009765625,116.60488626231319,34100.0,null,null,null,null,null,null,adsb,2025-10-04T05:34:22.180Z


In [0]:
df_flights_rank.display()

timestamp,flight_id,typecode,latitude,longitude,altitude,groundspeed,track,vertical_rate,mach,TAS,CAS,source,timestamp_converted
1758478407371000051,prc811906936,A359,39.23548294326006,116.59002685546875,32100.0,541.0,20.345217545191204,0.0,null,null,null,adsb,2025-09-21T18:13:27.371Z
1758478408653000116,prc811906936,A359,39.23789978027344,116.59121306046195,32100.0,540.0,20.48155454062644,0.0,null,null,null,adsb,2025-09-21T18:13:28.653Z
1758478409267999887,prc811906936,A359,39.2397657491393,116.59210205078125,32100.0,540.0,20.48155454062644,0.0,null,null,null,adsb,2025-09-21T18:13:29.267Z
1758478410394000053,prc811906936,A359,39.242233017743644,116.59326171875,32100.0,540.0,20.48155454062644,0.0,null,null,null,adsb,2025-09-21T18:13:30.394Z
1758478412645999908,prc811906936,A359,39.247467041015625,116.59581059994905,32100.0,null,null,null,null,null,null,adsb,2025-09-21T18:13:32.645Z
1758478414678999901,prc811906936,A359,39.252334853350106,116.59814453125,32100.0,null,null,null,null,null,null,adsb,2025-09-21T18:13:34.678Z
1758478415846999884,prc811906936,A359,39.254791259765625,116.59933338994566,32100.0,null,null,null,null,null,null,adsb,2025-09-21T18:13:35.846Z
1758478416693000078,prc811906936,A359,39.25685119628906,116.6002887228261,32100.0,null,null,null,null,null,null,adsb,2025-09-21T18:13:36.693Z
1758478417625000000,prc811906936,A359,39.25946044921875,116.60154259723166,32100.0,null,null,null,null,null,null,adsb,2025-09-21T18:13:37.625Z
1758478418232000113,prc811906936,A359,39.260760808395126,116.6021728515625,32100.0,540.0,20.382121011129733,0.0,null,null,null,adsb,2025-09-21T18:13:38.232Z


In [0]:
df_flights_train.display()

timestamp,flight_id,typecode,latitude,longitude,altitude,groundspeed,track,vertical_rate,mach,TAS,CAS,source,timestamp_converted
1749615988230000019,prc785525970,A359,33.63473665916313,-84.42375183105469,825.0936,145.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:28.230Z
1749615988789999962,prc785525970,A359,33.634735107421875,-84.42421426578443,825.0936,146.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:28.789Z
1749615989187000036,prc785525970,A359,33.634735107421875,-84.42455058195151,825.0936,148.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:29.187Z
1749615989861000061,prc785525970,A359,33.634735107421875,-84.42505505620215,825.0936,152.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:29.861Z
1749615991375000000,prc785525970,A359,33.634735107421875,-84.42640032087053,825.0936,156.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:31.375Z
1749615992947999954,prc785525970,A359,33.63473665916313,-84.42781448364258,825.0936,161.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:32.947Z
1749615994233999968,prc785525970,A359,33.634735107421875,-84.4288666394292,825.0936,166.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:34.233Z
1749615994617000103,prc785525970,A359,33.63473665916313,-84.42930221557617,825.0936,166.0,270.0,128.0,null,null,null,adsb,2025-06-11T04:26:34.617Z
1749615995242000103,prc785525970,A359,33.63473665916313,-84.429931640625,825.0936,169.0,270.0,192.0,null,null,null,adsb,2025-06-11T04:26:35.242Z
1749615995765000105,prc785525970,A359,33.634735107421875,-84.43043611487565,825.0936,170.0,270.0,192.0,null,null,null,adsb,2025-06-11T04:26:35.765Z


In [0]:
df_fuel_final.display()

idx,flight_id,start,end,fuel_kg,start_converted,end_converted
0,prc814299936,1759292304006000000,1759292558898000000,57.606184,2025-10-01T04:18:24.006Z,2025-10-01T04:22:38.898Z
1,prc814299936,1759292558898000000,1759292859176000000,66.224432,2025-10-01T04:22:38.898Z,2025-10-01T04:27:39.176Z
2,prc814299936,1759292859176000000,1759293098723000000,24.493968,2025-10-01T04:27:39.176Z,2025-10-01T04:31:38.723Z
3,prc814299936,1759293165388000000,1759293304979000000,8.164656,2025-10-01T04:32:45.388Z,2025-10-01T04:35:04.979Z
4,prc814299936,1759293304979000000,1759293420077000000,16.329312,2025-10-01T04:35:04.979Z,2025-10-01T04:37:00.077Z
5,prc814299936,1759293526174000000,1759293768729000000,33.112216,2025-10-01T04:38:46.174Z,2025-10-01T04:42:48.729Z
6,prc814299936,1759293834603000000,1759293877911000000,16.329312,2025-10-01T04:43:54.603Z,2025-10-01T04:44:37.911Z
7,prc814299936,1759294005315000000,1759294072079000000,8.164656,2025-10-01T04:46:45.315Z,2025-10-01T04:47:52.079Z
8,prc814299936,1759294072079000000,1759294144906000000,8.164656,2025-10-01T04:47:52.079Z,2025-10-01T04:49:04.906Z
9,prc814299936,1759294144906000000,1759294187341000000,24.94756,2025-10-01T04:49:04.906Z,2025-10-01T04:49:47.341Z


In [0]:
df_fuel_final_submit.display()

idx,flight_id,start,end,fuel_kg,start_converted,end_converted
0,prc806615763,1756695790925000000,1756696071584000000,null,2025-09-01T03:03:10.925Z,2025-09-01T03:07:51.584Z
1,prc806615763,1756696071584000000,1756696370921000000,null,2025-09-01T03:07:51.584Z,2025-09-01T03:12:50.921Z
2,prc806615763,1756696370921000000,1756696671404000000,null,2025-09-01T03:12:50.921Z,2025-09-01T03:17:51.404Z
3,prc806615763,1756696671404000000,1756696970539000000,null,2025-09-01T03:17:51.404Z,2025-09-01T03:22:50.539Z
4,prc806615763,1756696970539000000,1756697270727000000,null,2025-09-01T03:22:50.539Z,2025-09-01T03:27:50.727Z
5,prc806615763,1756697270727000000,1756697570802000000,null,2025-09-01T03:27:50.727Z,2025-09-01T03:32:50.802Z
6,prc806615763,1756697570802000000,1756697871778000000,null,2025-09-01T03:32:50.802Z,2025-09-01T03:37:51.778Z
7,prc806615763,1756697871778000000,1756698170706000000,null,2025-09-01T03:37:51.778Z,2025-09-01T03:42:50.706Z
8,prc806615763,1756698170706000000,1756698470452000000,null,2025-09-01T03:42:50.706Z,2025-09-01T03:47:50.452Z
9,prc806615763,1756698470452000000,1756698771183000000,null,2025-09-01T03:47:50.452Z,2025-09-01T03:52:51.183Z


In [0]:
df_fuel_rank.display()

idx,flight_id,start,end,fuel_kg,start_converted,end_converted
0,prc806615763,1756695790925000000,1756696071584000000,200.0,2025-09-01T03:03:10.925Z,2025-09-01T03:07:51.584Z
1,prc806615763,1756696071584000000,1756696370921000000,200.0,2025-09-01T03:07:51.584Z,2025-09-01T03:12:50.921Z
2,prc806615763,1756696370921000000,1756696671404000000,200.0,2025-09-01T03:12:50.921Z,2025-09-01T03:17:51.404Z
3,prc806615763,1756696671404000000,1756696970539000000,200.0,2025-09-01T03:17:51.404Z,2025-09-01T03:22:50.539Z
4,prc806615763,1756696970539000000,1756697270727000000,100.0,2025-09-01T03:22:50.539Z,2025-09-01T03:27:50.727Z
5,prc806615763,1756697270727000000,1756697570802000000,200.0,2025-09-01T03:27:50.727Z,2025-09-01T03:32:50.802Z
6,prc806615763,1756697570802000000,1756697871778000000,200.0,2025-09-01T03:32:50.802Z,2025-09-01T03:37:51.778Z
7,prc806615763,1756697871778000000,1756698170706000000,200.0,2025-09-01T03:37:51.778Z,2025-09-01T03:42:50.706Z
8,prc806615763,1756698170706000000,1756698470452000000,100.0,2025-09-01T03:42:50.706Z,2025-09-01T03:47:50.452Z
9,prc806615763,1756698470452000000,1756698771183000000,100.0,2025-09-01T03:47:50.452Z,2025-09-01T03:52:51.183Z


In [0]:
df_fuel_rank_submit.display()

idx,flight_id,start,end,fuel_kg,start_converted,end_converted
0,prc806615763,1756695790925000000,1756696071584000000,null,2025-09-01T03:03:10.925Z,2025-09-01T03:07:51.584Z
1,prc806615763,1756696071584000000,1756696370921000000,null,2025-09-01T03:07:51.584Z,2025-09-01T03:12:50.921Z
2,prc806615763,1756696370921000000,1756696671404000000,null,2025-09-01T03:12:50.921Z,2025-09-01T03:17:51.404Z
3,prc806615763,1756696671404000000,1756696970539000000,null,2025-09-01T03:17:51.404Z,2025-09-01T03:22:50.539Z
4,prc806615763,1756696970539000000,1756697270727000000,null,2025-09-01T03:22:50.539Z,2025-09-01T03:27:50.727Z
5,prc806615763,1756697270727000000,1756697570802000000,null,2025-09-01T03:27:50.727Z,2025-09-01T03:32:50.802Z
6,prc806615763,1756697570802000000,1756697871778000000,null,2025-09-01T03:32:50.802Z,2025-09-01T03:37:51.778Z
7,prc806615763,1756697871778000000,1756698170706000000,null,2025-09-01T03:37:51.778Z,2025-09-01T03:42:50.706Z
8,prc806615763,1756698170706000000,1756698470452000000,null,2025-09-01T03:42:50.706Z,2025-09-01T03:47:50.452Z
9,prc806615763,1756698470452000000,1756698771183000000,null,2025-09-01T03:47:50.452Z,2025-09-01T03:52:51.183Z


In [0]:
df_fuel_train.display()

idx,flight_id,start,end,fuel_kg,start_converted,end_converted
0,prc770822360,1744511464447000000,1744513264487000000,2500.0,2025-04-13T02:31:04.447Z,2025-04-13T03:01:04.487Z
1,prc770822360,1744513264487000000,1744514164226000000,1200.0000000000011,2025-04-13T03:01:04.487Z,2025-04-13T03:16:04.226Z
2,prc770822360,1744514164458000000,1744515964417000000,2500.0,2025-04-13T03:16:04.458Z,2025-04-13T03:46:04.417Z
3,prc770822360,1744515964417000000,1744516864540000000,1299.9999999999998,2025-04-13T03:46:04.417Z,2025-04-13T04:01:04.540Z
4,prc770822360,1744516864540000000,1744517764263000000,599.9999999999997,2025-04-13T04:01:04.540Z,2025-04-13T04:16:04.263Z
5,prc770831136,1744493433782000000,1744495119507000000,3049.952608,2025-04-12T21:30:33.782Z,2025-04-12T21:58:39.507Z
6,prc770831136,1744495119507000000,1744495360552000000,355.162536,2025-04-12T21:58:39.507Z,2025-04-12T22:02:40.552Z
7,prc770831136,1744495360552000000,1744496564881000000,1930.941144,2025-04-12T22:02:40.552Z,2025-04-12T22:22:44.881Z
8,prc770831136,1744496564881000000,1744497534572000000,1607.076456,2025-04-12T22:22:44.881Z,2025-04-12T22:38:54.572Z
9,prc770831136,1744497534572000000,1744497768575000000,370.131072,2025-04-12T22:38:54.572Z,2025-04-12T22:42:48.575Z



# Opis i przygotowanie danych – PRC Data Challenge 2025

**Projekt:** Linie lotnicze – efektywność paliwowa, wiek floty i szacowanie emisji CO₂

## 1. Źródło danych

Dane pochodzą z konkursu **PRC Data Challenge 2025 (EUROCONTROL)**. Zbiór łączy informacje o zużyciu paliwa pochodzące z raportów ACARS z trajektoriami lotów ADS-B udostępnionymi przez OpenSky Network.

Dane obejmują:
- Loty od kwietnia do października 2025 roku.
- 27 typów samolotów.
- 193 275 przedziałów zużycia paliwa.
- Parametry lotu, informacje o samolotach oraz lotniskach wylotu i przylotu.

**Celem projektu** jest analiza czynników wpływających na zużycie paliwa, porównanie efektywności paliwowej różnych typów samolotów, przybliżona analiza generacji floty oraz szacowanie emisji CO₂.

## 2. Wykorzystane tabele

| Tabela | Wykorzystana? | Uzasadnienie |
|---|---|---|
| `fuel_train`, `fuel_rank`, `fuel_final` | Tak | Zużycie paliwa (`fuel_kg`), czyli zmienna docelowa |
| `flights_train`, `flights_rank`, `flights_final` | Tak | Trajektorie ADS-B i pomiary ACARS, główne cechy objaśniające |
| `flightlist_train`, `flightlist_rank`, `flightlist_final` | Tak | Typ samolotu, lotniska i metadane lotów |
| `airports` | Tak | Współrzędne geograficzne i wysokość lotnisk |
| `fuel_rank_submit`, `fuel_final_submit` | Nie | Szablony zgłoszeń z pustym `fuel_kg`, bez dodatkowych informacji |

## 3. Zachowanie oryginalnych danych

Przed rozpoczęciem przetwarzania zapisujemy wszystkie dostępne kolumny i obserwacje z tabel źródłowych.

Nie usuwamy duplikatów ani nie próbkujemy trajektorii ADS-B i ACARS. Zachowujemy pełną rozdzielczość czasową pomiarów.

Dzięki temu możliwe jest ponowne wykorzystanie oryginalnych danych, obliczenie dodatkowych cech oraz przeprowadzenie bardziej szczegółowych analiz na późniejszych etapach projektu.

## 4. Łączenie danych

Dane łączymy w trzech etapach:

1. **Trajektorie i zużycie paliwa:** punkty trajektorii przypisujemy do odpowiednich przedziałów na podstawie `flight_id` oraz warunku `start ≤ timestamp < end`.
2. **Metadane lotów:** dołączamy wszystkie dostępne informacje z `flightlist` na podstawie `flight_id`, w tym typ samolotu i lotniska.
3. **Informacje o lotniskach:** dołączamy dane z `airports` na podstawie kodów ICAO lotniska wylotu i przylotu.

**Jednostką analizy jest przedział zużycia paliwa trwający od 5 do 60 minut, a nie cały lot.**

Dla każdego przedziału tworzymy jeden wiersz zawierający zagregowane parametry trajektorii, metadane lotu i informacje o zużyciu paliwa.

## 5. Obliczanie nowych cech

Dla każdego przedziału paliwowego wyznaczamy dodatkowe zmienne opisujące przebieg lotu.

### 5.1. Parametry lotu

Obliczamy średnią, minimum, maksimum oraz odchylenie standardowe dla:

- Wysokości lotu (`altitude`).
- Prędkości względem ziemi (`groundspeed`).
- Prędkości pionowej (`vertical_rate`).
- Liczby Macha (`mach`).
- Prędkości rzeczywistej (`TAS`).
- Prędkości kalibrowanej (`CAS`).
- Szerokości i długości geograficznej (`latitude`, `longitude`).
- Kierunku lotu (`track`).

### 5.2. Charakterystyka trajektorii

Wyznaczamy:

- `n_points` – liczbę punktów trajektorii w przedziale.
- `n_acars` – liczbę obserwacji ACARS.
- `n_adsb` – liczbę obserwacji ADS-B.
- `first_point_ns`, `last_point_ns` – znaczniki czasu pierwszego i ostatniego pomiaru.
- `alt_first`, `alt_last` – pierwszą i ostatnią zarejestrowaną wysokość.
- `lat_first`, `lat_last` – pierwszą i ostatnią szerokość geograficzną.
- `lon_first`, `lon_last` – pierwszą i ostatnią długość geograficzną.

### 5.3. Parametry czasowe i zmiana wysokości

Obliczamy:

- **Czas trwania przedziału:** `dur_s = (end - start) / 1e9`.
- **Czas od pierwszego dostępnego przedziału danego lotu:** `t_from_first_s`.
- **Zmianę wysokości:** `alt_change = alt_last - alt_first`.

Zmiana wysokości może pomóc w rozróżnianiu wznoszenia, zniżania i lotu na względnie stałym poziomie.

### 5.4. Dystans trasy

Na podstawie współrzędnych geograficznych lotniska wylotu i przylotu obliczamy odległość wielkiego koła za pomocą wzoru Haversine.

Wynik zapisujemy w kolumnie `route_km`.

Jest to odległość geograficzna między lotniskami, a nie rzeczywista długość przebytej trajektorii.

### 5.5. Szacowanie emisji CO₂

Emisję dwutlenku węgla szacujemy na podstawie zużycia paliwa:

`co2_kg = fuel_kg × 3.16`

Współczynnik 3,16 oznacza przybliżoną masę CO₂ powstającą podczas spalania 1 kg paliwa lotniczego.

Obliczona emisja jest wartością szacunkową, zależną od dokładności pomiarów zużycia paliwa.

## 6. Podział danych

Wykorzystujemy oryginalny podział czasowy udostępniony przez organizatorów konkursu.

| Zbiór | Okres | Przeznaczenie |
|---|---|---|
| `train` | 12 IV – 31 VIII 2025 | Trenowanie modeli |
| `rank` | 1–27 IX 2025 | Walidacja, dobór parametrów i porównanie modeli |
| `final` | 1–31 X 2025 | Końcowa ocena modeli (RMSE w kg) |

Podział czasowy lepiej odzwierciedla rzeczywiste zastosowanie modelu niż podział losowy.

Pozwala również ograniczyć ryzyko wycieku informacji pomiędzy zbiorami, ponieważ przedziały tego samego lotu nie są losowo rozdzielane między zbiory.

## 7. Wiek i generacja floty

Dane nie zawierają numerów rejestracyjnych samolotów, identyfikatorów `icao24` ani informacji o liniach lotniczych.

Z tego powodu nie możemy bezpośrednio obliczyć rzeczywistego wieku poszczególnych samolotów.

Jako przybliżenie wykorzystamy **generację typu samolotu**:

- **Nowsze generacje:** A20N, A21N, B38M, B39M.
- **Starsze generacje:** A320, A321, B738.

Podział ten umożliwi porównanie efektywności paliwowej nowszych i starszych konstrukcji.

Należy jednak pamiętać, że generacja modelu samolotu nie jest równoznaczna z rzeczywistym wiekiem konkretnego egzemplarza.

## 8. Zapis przygotowanych danych

Dane zapisujemy w formacie **Delta** w Databricks.

Powstaje łącznie 13 tabel:

| Rodzaj tabel | Liczba | Zawartość |
|---|---|---|
| `fuel_full_*` | 3 | Wszystkie dane dotyczące zużycia paliwa |
| `flights_full_*` | 3 | Wszystkie obserwacje trajektorii ADS-B i ACARS |
| `flightlist_full_*` | 3 | Wszystkie dostępne metadane lotów |
| `airports_full` | 1 | Informacje o lotniskach |
| `fuel_segments_*` | 3 | Zintegrowane dane analityczne |

Trzy główne tabele analityczne to:

- `fuel_segments_train` – zbiór treningowy.
- `fuel_segments_rank` – zbiór walidacyjny.
- `fuel_segments_final` – zbiór testowy.

Każdy wiersz tabel analitycznych odpowiada jednemu przedziałowi zużycia paliwa. Pełne pomiary trajektorii są przechowywane oddzielnie, aby zachować ich oryginalną szczegółowość.

## 9. Ograniczenia danych

Podczas interpretacji wyników uwzględniamy następujące ograniczenia:

- Dane obejmują tylko loty, dla których zebrano co najmniej 8 raportów ACARS, dlatego próba nie jest losowa.
- Przedziały krótsze niż 5 minut i dłuższe niż 60 minut zostały wykluczone przez organizatorów.
- Parametry `mach`, `TAS` i `CAS` występują głównie w rzadszych pomiarach ACARS, a w obserwacjach ADS-B są puste.
- Zużycie paliwa (`fuel_kg`) obliczono na podstawie różnic raportowanego paliwa na pokładzie, co może wprowadzać szum pomiarowy.
- Nie znamy rzeczywistego wieku samolotów ani ich przynależności do linii lotniczych.
- Dystans `route_km` jest odległością między lotniskami, a nie długością rzeczywistej trasy lotu.
- Emisja CO₂ jest szacowana na podstawie współczynnika emisji, a nie mierzona bezpośrednio.

## 10. Efekt końcowy i dalsze kroki

W wyniku przetwarzania otrzymujemy trzy zintegrowane zbiory analityczne, zawierające:

- Zużycie paliwa (`fuel_kg`).
- Szacowaną emisję CO₂ (`co2_kg`).
- Zagregowane parametry trajektorii lotu.
- Charakterystykę czasową przedziałów.
- Typ samolotu i metadane lotu.
- Informacje o lotniskach oraz dystansie trasy.

Wszystkie oryginalne dane pozostają dostępne w osobnych tabelach.

**Na tym etapie nie przeprowadzamy jeszcze selekcji cech ani trenowania modeli.**

Kolejne etapy projektu obejmą eksploracyjną analizę danych (EDA), analizę braków i rozkładów zmiennych, porównanie efektywności paliwowej typów i generacji samolotów oraz budowę modeli przewidujących zużycie paliwa.

Zmienne `fuel_kg` i `co2_kg` są ze sobą bezpośrednio powiązane, dlatego szacowana emisja CO₂ nie będzie wykorzystywana jako cecha wejściowa modelu przewidującego zużycie paliwa.


In [0]:
# 1. FUNKCJA BUDUJĄCA ZBIÓR DANYCH


def build(phase, fuel, flights, flightlist, airports):
    # 1. ZAPIS WSZYSTKICH ORYGINALNYCH DANYCH

    for name, df in [
        ("fuel", fuel),
        ("flights", flights),
        ("flightlist", flightlist),
    ]:
        (
            df.write.mode("overwrite")
            .format("delta")
            .saveAsTable(f"{CATALOG}.{SCHEMA}.{name}_full_{phase}")
        )

    # Lotniska są wspólne dla wszystkich faz.
    # Zachowujemy wszystkie ich oryginalne kolumny.

    if phase == "train":
        (
            airports.write.mode("overwrite")
            .format("delta")
            .saveAsTable(f"{CATALOG}.{SCHEMA}.airports_full")
        )

    # 2. TRAJEKTORIE

    fl = flights.withColumn("is_acars", F.lower(F.col("source")) == "acars").withColumn(
        "is_adsb", F.lower(F.col("source")) == "adsb"
    )

    # 3. SPRAWDZENIE UNIKALNOŚCI IDX

    duplicate_idx = (
        fuel.groupBy("idx").count().filter(F.col("count") > 1).limit(1).count()
    )

    if duplicate_idx:
        raise ValueError(f"Faza {phase}: kolumna idx nie jest unikalna.")

    # 4. PRZYPISANIE PUNKTÓW TRAJEKTORII DO PRZEDZIAŁÓW

    joined = fuel.alias("f").join(
        fl.alias("t"),
        (
            (F.col("f.flight_id") == F.col("t.flight_id"))
            & (F.col("t.timestamp") >= F.col("f.start"))
            & (F.col("t.timestamp") < F.col("f.end"))
        ),
        "left",
    )

    # 5. FUNKCJA STATYSTYK DLA KOLUMN LICZBOWYCH

    # NaN zamieniamy na NULL wyłącznie do agregacji.
    # Oryginalne pomiary pozostają zapisane bez zmian.

    def stats(col_name, prefix):
        c = F.col(f"t.{col_name}").cast("double")

        clean = F.when(F.isnan(c), F.lit(None)).otherwise(c)

        return [
            F.avg(clean).alias(f"{prefix}_mean"),
            F.min(clean).alias(f"{prefix}_min"),
            F.max(clean).alias(f"{prefix}_max"),
            F.stddev(clean).alias(f"{prefix}_std"),
        ]

    # 6. AGREGACJE TRAJEKTORII

    aggregations = []

    numeric_columns = {
        "altitude": "alt",
        "groundspeed": "gs",
        "vertical_rate": "vr",
        "mach": "mach",
        "TAS": "tas",
        "CAS": "cas",
        "latitude": "lat",
        "longitude": "lon",
        "track": "track",
    }

    for col_name, prefix in numeric_columns.items():
        aggregations.extend(stats(col_name, prefix))

    aggregations.extend(
        [
            # Liczba punktów trajektorii
            F.count("t.timestamp").alias("n_points"),
            # Liczba punktów ACARS
            F.sum(F.when(F.col("t.is_acars") == True, 1).otherwise(0)).alias("n_acars"),
            # Liczba punktów ADS-B
            F.sum(F.when(F.col("t.is_adsb") == True, 1).otherwise(0)).alias("n_adsb"),
            # Pierwszy i ostatni timestamp
            F.min("t.timestamp").alias("first_point_ns"),
            F.max("t.timestamp").alias("last_point_ns"),
            # Wysokość na początku i końcu
            F.min_by(F.col("t.altitude"), F.col("t.timestamp")).alias("alt_first"),
            F.max_by(F.col("t.altitude"), F.col("t.timestamp")).alias("alt_last"),
            # Szerokość geograficzna
            F.min_by(F.col("t.latitude"), F.col("t.timestamp")).alias("lat_first"),
            F.max_by(F.col("t.latitude"), F.col("t.timestamp")).alias("lat_last"),
            # Długość geograficzna
            F.min_by(F.col("t.longitude"), F.col("t.timestamp")).alias("lon_first"),
            F.max_by(F.col("t.longitude"), F.col("t.timestamp")).alias("lon_last"),
        ]
    )

    # 7. AGREGACJA DLA KAŻDEGO SEGMENTU

    seg_stats = joined.groupBy(F.col("f.idx").alias("idx")).agg(*aggregations)

    # 8. WSZYSTKIE KOLUMNY FUEL + AGREGATY

    seg = (
        fuel.join(seg_stats, "idx", "left")
        # Czas trwania segmentu w sekundach
        .withColumn("dur_s", (F.col("end") - F.col("start")) / 1e9)
        # Czas od początku pierwszego segmentu lotu
        .withColumn(
            "t_from_first_s",
            (F.col("start") - F.min("start").over(Window.partitionBy("flight_id")))
            / 1e9,
        )
        # Zmiana wysokości w segmencie
        .withColumn(
            "alt_change",
            F.col("alt_last").cast("double") - F.col("alt_first").cast("double"),
        )
    )

    # 9. WSZYSTKIE METADANE LOTU
    # Zachowujemy wszystkie kolumny flightlist.
    # Prefiks flight_ zapobiega konfliktom nazw.
    meta = flightlist.select(
        F.col("flight_id"),
        *[
            F.col(c).alias(f"flight_{c}")
            for c in flightlist.columns
            if c != "flight_id"
        ],
    )

    # 10. WSZYSTKIE KOLUMNY LOTNISK
    # Lotnisko początkowe
    ap_o = airports.select(
        *[F.col(c).alias(f"origin_airport_{c}") for c in airports.columns]
    )

    # Lotnisko docelowe
    ap_d = airports.select(
        *[F.col(c).alias(f"destination_airport_{c}") for c in airports.columns]
    )

    # 11. DOŁĄCZENIE LOTNISK DO METADANYCH
    meta = meta.join(
        ap_o, F.col("flight_origin_icao") == F.col("origin_airport_icao"), "left"
    ).join(
        ap_d,
        F.col("flight_destination_icao") == F.col("destination_airport_icao"),
        "left",
    )

    # 12. DYSTANS TRASY - HAVERSINE
    lat1 = F.radians(F.col("origin_airport_latitude").cast("double"))
    lat2 = F.radians(F.col("destination_airport_latitude").cast("double"))
    dlat = lat2 - lat1
    dlon = F.radians(
        F.col("destination_airport_longitude").cast("double")
        - F.col("origin_airport_longitude").cast("double")
    )
    a = F.pow(F.sin(dlat / 2), 2) + F.cos(lat1) * F.cos(lat2) * F.pow(
        F.sin(dlon / 2), 2
    )

    # Zabezpieczenie przed błędami numerycznymi
    a_safe = F.least(F.lit(1.0), F.greatest(F.lit(0.0), a))
    meta = meta.withColumn("route_km", 6371.0 * 2 * F.asin(F.sqrt(a_safe)))

    # 13. ŁĄCZENIE WSZYSTKICH DANYCH
    out = seg.join(meta, "flight_id", "left")

    # 14. EMISJA CO2
    # Przyjmujemy 3.16 kg CO2 / kg paliwa.

    out = out.withColumn("co2_kg", F.col("fuel_kg").cast("double") * F.lit(3.16))

    # 15. ZAPIS KOŃCOWEGO ZBIORU DO UNITY CATALOG
    (
        out.write.mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.fuel_segments_{phase}")
    )

    print(f"Zapisano: {CATALOG}.{SCHEMA}.fuel_segments_{phase}")

    return out





In [0]:
# 16. PRZETWARZANIE TRAIN / RANK / FINAL
train = build(
    "train", df_fuel_train, df_flights_train, df_flightlist_train, df_airports
)
valid = build("rank", df_fuel_rank, df_flights_rank, df_flightlist_rank, df_airports)
test = build("final", df_fuel_final, df_flights_final, df_flightlist_final, df_airports)



Zapisano: pk_big_data.curated.fuel_segments_train
Zapisano: pk_big_data.curated.fuel_segments_rank
Zapisano: pk_big_data.curated.fuel_segments_final


In [0]:
# 17. WERYFIKACJA WYNIKÓW
for phase in ["train", "rank", "final"]:
    table_name = f"{CATALOG}.{SCHEMA}.fuel_segments_{phase}"

    df_check = spark.table(table_name)
    print(f"{phase}: {df_check.count()} wierszy, {len(df_check.columns)} kolumn")


train: 131530 wierszy, 75 kolumn
rank: 24289 wierszy, 75 kolumn
final: 37456 wierszy, 75 kolumn


In [0]:
# 18. PODGLĄD DANYCH TRENINGOWYCH
display(spark.table(f"{CATALOG}.{SCHEMA}.fuel_segments_train").limit(20))

flight_id,idx,start,end,fuel_kg,start_converted,end_converted,alt_mean,alt_min,alt_max,alt_std,gs_mean,gs_min,gs_max,gs_std,vr_mean,vr_min,vr_max,vr_std,mach_mean,mach_min,mach_max,mach_std,tas_mean,tas_min,tas_max,tas_std,cas_mean,cas_min,cas_max,cas_std,lat_mean,lat_min,lat_max,lat_std,lon_mean,lon_min,lon_max,lon_std,track_mean,track_min,track_max,track_std,n_points,n_acars,n_adsb,first_point_ns,last_point_ns,alt_first,alt_last,lat_first,lat_last,lon_first,lon_last,dur_s,t_from_first_s,alt_change,flight_flight_date,flight_aircraft_type,flight_takeoff,flight_landed,flight_origin_icao,flight_origin_name,flight_destination_icao,flight_destination_name,origin_airport_icao,origin_airport_longitude,origin_airport_latitude,origin_airport_elevation,destination_airport_icao,destination_airport_longitude,destination_airport_latitude,destination_airport_elevation,route_km,co2_kg
prc770822360,0,1744511464447000000,1744513264487000000,2500.0,2025-04-13T02:31:04.447Z,2025-04-13T03:01:04.487Z,35988.58500078253,35974.9344,35999.928,12.44544527391049,472.013723150358,465.0,480.0,4.0544565027709,-3.360381861575179,-64.0,64.0,46.37252357699412,0.86,0.86,0.86,null,null,null,null,null,null,null,null,null,46.237736875519644,45.18333333333333,47.22285461425781,0.6103391099430924,21.883091492274225,19.505882263183594,24.35,1.4556952989518255,301.22856397293026,300.1515479818221,302.29205517325994,0.560419024596875,3077,1,3076,1744511464447000000,1744513264184999943,null,35999.928,45.18333333333333,47.22285461425781,24.35,19.505882263183594,1800.04,0.0,null,2025-04-12,B789,2025-04-12T16:15:35.000,2025-04-13T04:33:52.000,WMKK,Kuala Lumpur International Airport,EHAM,Amsterdam Schiphol Airport,WMKK,101.709999,2.74558,69,EHAM,4.76389,52.308601,-11,10236.638941091825,7900.0
prc770822360,2,1744514164458000000,1744515964417000000,2500.0,2025-04-13T03:16:04.458Z,2025-04-13T03:46:04.417Z,35988.99060517793,35974.9344,35999.928,12.400906479713907,511.88579387186627,493.0,525.0,9.377808426969006,-36.040854224698236,-128.0,64.0,43.789453207161394,0.86,0.86,0.86,null,null,null,null,null,null,null,null,null,49.48169800519137,48.233333333333334,50.713531494140625,0.7165572021706532,14.338007378866664,11.638307313661318,16.95,1.5286036074014282,305.6584113839573,303.9986040132782,307.34229030628615,0.7405114823611789,3479,1,3478,1744514164458000000,1744515963571000099,null,35999.928,48.233333333333334,50.713531494140625,16.95,11.638307313661318,1799.959,2700.011,null,2025-04-12,B789,2025-04-12T16:15:35.000,2025-04-13T04:33:52.000,WMKK,Kuala Lumpur International Airport,EHAM,Amsterdam Schiphol Airport,WMKK,101.709999,2.74558,69,EHAM,4.76389,52.308601,-11,10236.638941091825,7900.0
prc770822360,1,1744513264487000000,1744514164226000000,1200.0000000000011,2025-04-13T03:01:04.487Z,2025-04-13T03:16:04.226Z,35989.29034362479,35974.9344,35999.928,12.361191419446147,484.7035490605428,475.0,497.0,7.434830617053677,-29.862212943632567,-64.0,64.0,36.801336411330745,0.86,0.86,0.86,null,null,null,null,null,null,null,null,null,47.7217855326957,47.21666666666667,48.24165344238281,0.28755524028346957,18.226099218672655,16.94908728966346,19.516666666666666,0.7411761469703806,300.68957777902773,298.80569035120664,306.6150454751361,2.521644394838623,1789,1,1788,1744513264487000000,1744514163982000113,null,35974.9344,47.21666666666667,48.24165344238281,19.516666666666666,16.94908728966346,899.739,1800.04,null,2025-04-12,B789,2025-04-12T16:15:35.000,2025-04-13T04:33:52.000,WMKK,Kuala Lumpur International Airport,EHAM,Amsterdam Schiphol Airport,WMKK,101.709999,2.74558,69,EHAM,4.76389,52.308601,-11,10236.638941091825,3792.0000000000036
prc770822360,3,1744515964417000000,1744516864540000000,1299.9999999999998,2025-04-13T03:46:04.417Z,2025-04-13T04:01:04.540Z,35988.4749830669,35949.940800000004,36024.9216,13.181922965468585,506.3889816360601,499.0,515.0,4.1066336454252585,-34.45742904841402,-256.0,128.0,64.97460032545939,0.86,0.86,0.86,null,null,null,null,null,null,null,null,null,51.2429791